# Fit AGC to Illuminance: Processing Notebook

This notebook is the formal Python processing step for the AGC-to-illuminance analysis. It gathers raw FLIC recordings, runs the empirical AGC-kernel fit, writes reusable cached processing data, and produces the linear-scale MATLAB input file used by `fitEmpircalAGCtoIlluminance.m`.


## Setup

Load the analysis utility module and notebook-only helpers. The utility owns the path defaults for this directory, including `cached_processing_data/` and `camera_agc_illuminance_linear_scale.mat`.


In [ ]:
import importlib
import os

from natsort import natsorted

import fit_agc_to_illuminance_util


## Select Recordings

Collect the raw `GKA` recording directories that will enter the processing fit. The current notebook preserves the existing exploratory scope of the run by processing the first four valid subject folders in the 2026 scripted indoor/outdoor dataset while skipping `FLIC_18`. Adjust `maximum_subjects_to_process` or the filtering rules here when intentionally regenerating the cache from a different recording set.


In [ ]:
flic_raw_path: str = "/Volumes/FLIC_raw/NEWscriptedIndoorOutdoorVideos2026"
maximum_subjects_to_process: int = 4
subjects_to_skip: set[str] = {"FLIC_18"}

recording_paths: list[str] = []
valid_subject_count: int = 0

for subject_dir in natsorted(os.listdir(flic_raw_path)):
    if valid_subject_count >= maximum_subjects_to_process:
        break

    if subject_dir in subjects_to_skip or subject_dir.startswith("."):
        continue

    subject_dir_path: str = os.path.join(flic_raw_path, subject_dir)
    if not os.path.isdir(subject_dir_path):
        continue

    for activity in natsorted(os.listdir(subject_dir_path)):
        if activity.startswith("."):
            continue

        activity_path: str = os.path.join(subject_dir_path, activity, "GKA")
        assert os.path.exists(activity_path), (
            f"Path does not exist: {activity_path}"
        )
        recording_paths.append(activity_path)

    valid_subject_count += 1

print(f"Selected {len(recording_paths)} recordings from {valid_subject_count} subjects.")


## Run Empirical AGC-Kernel Processing Fit

This step performs the expensive processing fit. It converts minispect counts to illuminance, applies the empirical AGC temporal kernel, estimates the shared lag, measures video saturation, builds per-recording diagnostics, and writes reusable cached data. The cache and diagnostic archives are intentionally kept under `cached_processing_data/`.


In [ ]:
importlib.reload(fit_agc_to_illuminance_util)

frame_saturation_data_path = (
    fit_agc_to_illuminance_util.CACHED_PROCESSING_DATA_DIR
    / "frame_saturation_calibration_data.npz"
)

summary_table = fit_agc_to_illuminance_util.fit_agc_to_illuminance(
    recording_paths,
    illuminance_diagnostics=True,
    illuminance_diagnostics_output_dir=(
        fit_agc_to_illuminance_util.ILLUMINANCE_DIAGNOSTICS_OUTPUT_DIR
    ),
    frame_saturation_data_output_path=frame_saturation_data_path,
)

summary_table


## Generate Final Linear-Scale MATLAB Input

Reload the cached point cloud, apply the current point-selection filters, render the consolidated calibration-selection dashboard, and write `camera_agc_illuminance_linear_scale.mat`. This `.mat` file is the formal input to the MATLAB piecewise log-log fit.


In [ ]:
saturation_threshold_percent: float = 40.0
minimum_model_correlation: float = 0.9
initial_samples_to_exclude: int = 100

calibration_dashboard_axes, activity_highlight_axes = (
    fit_agc_to_illuminance_util.plot_frame_saturation_from_processed_data(
        processed_data_path=frame_saturation_data_path,
        maximum_saturation_percent=saturation_threshold_percent,
        minimum_correlation=minimum_model_correlation,
        initial_samples_to_exclude=initial_samples_to_exclude,
    )
)

fit_agc_to_illuminance_util.LINEAR_SCALE_MAT_OUTPUT_PATH


## Run MATLAB Piecewise Log-Log Fit

Run the MATLAB fitting script against the `camera_agc_illuminance_linear_scale.mat` file generated above. This is the final model-fitting step that reports the piecewise log-log conversion from camera AGC product to implied illuminance.


In [ ]:
# -----------------------------------------------------------------------------
# MATLAB final fit
# -----------------------------------------------------------------------------
# This cell intentionally runs the MATLAB script from Python so the notebook
# documents the full AGC-to-illuminance workflow end to end:
#
#   1. Python generates cached processing data and the linear-scale .mat file.
#   2. MATLAB loads that .mat file and fits the final piecewise log-log model.
#
# The script is called by file path because the repository file is currently
# named fitEmpircalAGCtoIlluminance.m. If the file is later renamed to the
# intended fitEmpiricalAGCToIlluminance.m spelling, update only this path.
# -----------------------------------------------------------------------------

import matlab.engine

matlab_fit_script = (
    fit_agc_to_illuminance_util.FIT_AGC_TO_ILLUMINANCE_DIR
    / "fitEmpircalAGCtoIlluminance.m"
)

matlab_engine = matlab.engine.start_matlab()
matlab_engine.cd(
    str(fit_agc_to_illuminance_util.FIT_AGC_TO_ILLUMINANCE_DIR),
    nargout=0,
)
matlab_engine.run(str(matlab_fit_script), nargout=0)
